In [1]:
import os

In [2]:
print(os.getcwd())

c:\Users\Sonia\documents\json-api-geocodage\notebooks


In [4]:
import os

print("je m'execute depuis :", os.getcwd())
print("le dossier data existe :", os.path.isdir("../data"))
print("le fichier existe     :", os.path.isfile("../data/restaurants.json"))
print("contenu de ../data    :", os.listdir("../data"))

je m'execute depuis : c:\Users\Sonia\documents\json-api-geocodage\notebooks
le dossier data existe : True
le fichier existe     : True
contenu de ../data    : ['restaurants.json']


In [5]:
from pathlib import Path
print(Path.cwd())

c:\Users\Sonia\documents\json-api-geocodage\notebooks


In [ ]:
# 1.2 Le lire : json.load
import json

with open("../data/restaurants.json", encoding="utf-8") as f:
    data = json.load(f)

print(type(data))
print(data["ville"], "-", data["pays"])
print("nombre de restaurants :", len(data["restaurants"]))

<class 'dict'>
Catane - Italie
nombre de restaurants : 4


In [ ]:
# 1.3 Descendre dans l'imbrication
print(data["restaurants"][0]["nom"])
print(data["restaurants"][0]["notes"][0])

Gelateria Zio Pietro dal 1964
4.6


In [9]:
#1.4 Parcourir
for r in data["restaurants"]:
    moyenne = sum(r["notes"]) / len(r["notes"])
    print(f"{r['nom']:32} {moyenne:.2f}")
    

Gelateria Zio Pietro dal 1964    4.70
A Casa d'Amici                   4.45
La Bitta                         4.15
Steak House                      4.00


In [10]:
#1.5 L'écrire : json.dumps
#L'opération inverse. dumps transforme un objet Python en chaîne JSON.

print(json.dumps(data["restaurants"][1], ensure_ascii=False, indent=2))

{
  "nom": "A Casa d'Amici",
  "adresse": "Via Fischetti, 14, 95131 Catania CT",
  "type": "trattoria",
  "notes": [
    4.4,
    4.5
  ]
}


In [11]:
#2. Du JSON au DataFrame
#2.1 json_normalize, et la surprise

import pandas as pd

df = pd.json_normalize(data)
print(df.shape)
print(df.columns.tolist())

(1, 4)
['ville', 'pays', 'releve', 'restaurants']


In [12]:
#2.2 record_path : dire où sont les lignes
df = pd.json_normalize(data, record_path="restaurants")
print(df)

                             nom                                   adresse  \
0  Gelateria Zio Pietro dal 1964  Via Porta di Ferro, 47, 95131 Catania CT   
1                 A Casa d'Amici       Via Fischetti, 14, 95131 Catania CT   
2                       La Bitta    Via Acquicella Porto, 95121 Catania CT   
3                    Steak House   Via Porta di Ferro, 8, 95100 Catania CT   

        type                 notes  
0    glacier       [4.6, 4.8, 4.7]  
1  trattoria            [4.4, 4.5]  
2    poisson  [4.2, 4.0, 4.3, 4.1]  
3      grill            [3.9, 4.1]  


In [13]:
#2.3 meta : remonter chercher le contexte
#En descendant dans restaurants, tu as perdu la ville et le pays. meta les récupère.

df = pd.json_normalize(data, record_path="restaurants", meta=["ville", "pays"])
print(df)

                             nom                                   adresse  \
0  Gelateria Zio Pietro dal 1964  Via Porta di Ferro, 47, 95131 Catania CT   
1                 A Casa d'Amici       Via Fischetti, 14, 95131 Catania CT   
2                       La Bitta    Via Acquicella Porto, 95121 Catania CT   
3                    Steak House   Via Porta di Ferro, 8, 95100 Catania CT   

        type                 notes   ville    pays  
0    glacier       [4.6, 4.8, 4.7]  Catane  Italie  
1  trattoria            [4.4, 4.5]  Catane  Italie  
2    poisson  [4.2, 4.0, 4.3, 4.1]  Catane  Italie  
3      grill            [3.9, 4.1]  Catane  Italie  


In [19]:
import json

with open("../data/villes.json", encoding="utf-8") as f:
    data = json.load(f)

print(type(data))
print(data["pays"], "-", data["releve"])
print("nombre de villes :", len(data["villes"]))

<class 'dict'>
Italie - 2026-11-05
nombre de villes : 3


In [ ]:
#Nom de la 2eme ville et nb de restaurants
print(data["villes"][1]["nom"])
print("nombre de restaurants :", len(data["villes"][1]["restaurants"]))

Palerme
nombre de restaurants : 2


In [24]:
print(data.keys())
print(data["villes"][1].keys())

dict_keys(['pays', 'releve', 'villes'])
dict_keys(['nom', 'region', 'habitants', 'restaurants'])


In [28]:
#Nb de restaurants à "Syracuse"
for ville in data["villes"]:
    if ville["nom"] == "Syracuse":
        print("nombre de restaurants :", len(ville["restaurants"]))

nombre de restaurants : 2


In [29]:
#Nom du premier restaurant de Palerme
for ville in data["villes"]:
    if ville["nom"] == "Palerme":
        print(ville["restaurants"][0]["nom"])

Antica Focacceria


In [30]:
# Avec une boucle, affiche pour chaque ville son nom et son nombre de restaurants
for ville in data["villes"]:
    print(f"{ville['nom']} : {len(ville['restaurants'])} restaurants")

Catane : 3 restaurants
Palerme : 2 restaurants
Syracuse : 2 restaurants


Étape 3 — tout mettre à plat

Construis un DataFrame d'une ligne par restaurant — sept lignes au total — qui contienne aussi le nom de la ville, sa région et son nombre d'habitants.

In [31]:
import pandas as pd

df = pd.json_normalize(data)
print(df.shape)
print(df.columns.tolist())

(1, 3)
['pays', 'releve', 'villes']


In [33]:
df = pd.json_normalize(data, record_path="villes")
print(df)

        nom  region  habitants  \
0    Catane  Sicile     298762   
1   Palerme  Sicile     630828   
2  Syracuse  Sicile     121605   

                                         restaurants  
0  [{'nom': 'Gelateria Zio Pietro dal 1964', 'typ...  
1  [{'nom': 'Antica Focacceria', 'type': 'trattor...  
2  [{'nom': 'Don Camillo', 'type': 'poisson', 'no...  


In [34]:
df = pd.json_normalize(data, record_path="villes", meta=["pays", "releve", "villes"])
print(df)

        nom  region  habitants  \
0    Catane  Sicile     298762   
1   Palerme  Sicile     630828   
2  Syracuse  Sicile     121605   

                                         restaurants    pays      releve  \
0  [{'nom': 'Gelateria Zio Pietro dal 1964', 'typ...  Italie  2026-11-05   
1  [{'nom': 'Antica Focacceria', 'type': 'trattor...  Italie  2026-11-05   
2  [{'nom': 'Don Camillo', 'type': 'poisson', 'no...  Italie  2026-11-05   

                                              villes  
0  [{'nom': 'Catane', 'region': 'Sicile', 'habita...  
1  [{'nom': 'Catane', 'region': 'Sicile', 'habita...  
2  [{'nom': 'Catane', 'region': 'Sicile', 'habita...  


In [47]:
#Il faut d’abord calculer la moyenne des notes car c'est une liste chaque restaurant
import numpy as np

# Moyenne des notes de chaque restaurant
df_restos["note_resto"] = df_restos["notes"].apply(
    lambda n: np.mean(n) if isinstance(n, list) and len(n) > 0 else n
)

# Note moyenne par ville
df_villes["note_moyenne"] = (
    df_villes["nom"].map(df_restos.groupby("ville")["note_resto"].mean()).round(2)
)
print(df_villes)

        nom  nb_restaurants  note_moyenne
0    Catane               3          4.43
1   Palerme               2          4.41
2  Syracuse               2          4.47


In [ ]:
#"notes" n’est pas un nombre mais une liste (ex.[4, 5, 3]).
# #Pandas ne peut pas calculer une moyenne sur des listes. Vérifie-le avec :
print(df_restos["notes"].head())

0         [4.6, 4.8, 4.7]
1              [4.4, 4.5]
2    [4.2, 4.0, 4.3, 4.1]
3         [4.3, 4.1, 4.4]
4              [4.5, 4.6]
Name: notes, dtype: object


In [48]:
#Ville avec la meilleur moyenne et combien de restaurants de chaque type, tous lieux confondus ?
meilleure = df_villes.loc[df_villes["note_moyenne"].idxmax()]
print(meilleure["nom"], ":", meilleure["note_moyenne"])

print(df_restos["type"].value_counts())

Syracuse : 4.47
type
trattoria    3
glacier      2
poisson      2
Name: count, dtype: int64
